# RAG Playground with Milvus Lite

---
## Phase 0 — Setup

> ### 🎯 Goal of Phase 0
> Get the environment ready. By the end of this phase you will have:
> 1. Installed the two libraries we need (`pymilvus` for the vector DB, `sentence-transformers` for embeddings).
> 2. Uploaded your own 2–3 markdown (`.md`) files into Colab.
>
> Nothing conceptual here yet — this is just plumbing so the later phases run smoothly.


### 0.1 — Install libraries
`pymilvus` includes **Milvus Lite** automatically (the embedded mode). Run this once per session.
It may take a minute. You can ignore dependency-resolver warnings.

In [ ]:
!pip install -q -U "pymilvus[milvus_lite]" sentence-transformers langchain langchain-community langchain-milvus langchain-huggingface
print("✅ Installed. If you see a 'restart runtime' prompt, you can ignore it for this notebook.")

### 0.2 — (Optional) Check for GPU
A GPU makes embedding faster but is **not required** for a few small docs.
To enable: menu → Runtime → Change runtime type → Hardware accelerator → **T4 GPU** → Save, then re-run Phase 0.

In [ ]:
import torch
print("GPU available:", torch.cuda.is_available())
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
print("Using device:", DEVICE)

### 0.3 — Upload your markdown files
Run this cell, then use the file picker to select your 2–3 `.md` files.
They get saved into a `docs/` folder inside Colab.

In [ ]:
import os
from google.colab import files

os.makedirs("docs", exist_ok=True)
print("Select your .md files in the picker below...")
uploaded = files.upload()

for name, content in uploaded.items():
    with open(os.path.join("docs", name), "wb") as f:
        f.write(content)

print("\n✅ Saved these files into docs/:")
for name in os.listdir("docs"):
    print("  -", name)

---
## Phase 1 — The raw RAG pipeline (no LangChain yet)

> ### 🎯 Goal of Phase 1
> See the **entire retrieval pipeline** with your own eyes, step by step, using raw
> `pymilvus`. No magic, no framework hiding the steps. By the end you will understand
> the five core moves of RAG retrieval:
>
> 1. **Load** — read the raw text out of your markdown files.
> 2. **Chunk** — split long text into smaller pieces (here: simple fixed-size).
> 3. **Embed** — turn each chunk into a vector (list of numbers) using a model.
> 4. **Store** — put those vectors into Milvus.
> 5. **Query** — embed your *question* the same way, ask Milvus for the closest chunks,
>    and read them back with their distance scores.
>
> This is the "aha" moment — you type a question and watch relevant chunks come back.


### 1.1 — Load the markdown files
We just read each file's text into memory. No processing yet.

In [ ]:
import os, glob

def load_markdown(folder="docs"):
    docs = []
    for path in glob.glob(os.path.join(folder, "*.md")):
        with open(path, "r", encoding="utf-8") as f:
            docs.append({"source": os.path.basename(path), "text": f.read()})
    return docs

documents = load_markdown()
print(f"Loaded {len(documents)} documents:")
for d in documents:
    print(f"  - {d['source']}: {len(d['text'])} characters")

### 1.2 — Chunk the text (simple fixed-size)
The simplest possible chunker: cut the text every `CHUNK_SIZE` characters.
We attach metadata to each chunk (which file it came from, its position) — this
becomes important in the schema phase.

👉 **This is your first experiment knob.** Change `CHUNK_SIZE` and re-run to see chunks change.

In [ ]:
CHUNK_SIZE = 500   # 👈 experiment: try 200, 500, 1000

def chunk_fixed(text, size):
    return [text[i:i + size] for i in range(0, len(text), size)]

chunks = []
for doc in documents:
    for i, piece in enumerate(chunk_fixed(doc["text"], CHUNK_SIZE)):
        chunks.append({
            "text": piece,
            "source": doc["source"],
            "chunk_index": i,
        })

print(f"Created {len(chunks)} chunks (CHUNK_SIZE={CHUNK_SIZE}).")
print("\n--- Example chunk ---")
print(chunks[0]["text"][:300], "...")

### 1.3 — Embed the chunks
Load a small, fast embedding model and convert every chunk's text into a vector.
`all-MiniLM-L6-v2` outputs **384-dim** vectors — remember that number, Milvus needs it.

👉 **This is your embedding knob** (you'll play with it in Phase 4).

In [ ]:
from sentence_transformers import SentenceTransformer

EMBEDDING_MODEL = "all-MiniLM-L6-v2"   # 👈 experiment later: "BAAI/bge-small-en-v1.5", "all-mpnet-base-v2"

embedder = SentenceTransformer(EMBEDDING_MODEL, device=DEVICE)
DIMENSION = embedder.get_sentence_embedding_dimension()
print(f"Model '{EMBEDDING_MODEL}' → vector dimension = {DIMENSION}")

chunk_texts = [c["text"] for c in chunks]
chunk_vectors = embedder.encode(chunk_texts, show_progress_bar=True)
print("Embedded", len(chunk_vectors), "chunks. Each vector has", len(chunk_vectors[0]), "numbers.")

### 1.4 — Store in Milvus Lite
Passing a **filename** to `MilvusClient` triggers Milvus Lite (embedded) mode.
We use the **quick-setup** API here (Milvus builds a minimal schema for us automatically).
We'll open up the schema ourselves in Phase 3.

In [ ]:
from pymilvus import MilvusClient

client = MilvusClient("milvus_demo.db")   # the .db file = Milvus Lite (embedded)

COLLECTION = "rag_chunks"
if client.has_collection(COLLECTION):
    client.drop_collection(COLLECTION)   # clean slate each run

client.create_collection(
    collection_name=COLLECTION,
    dimension=DIMENSION,   # MUST match the embedding model's output size
)

data = []
for i, c in enumerate(chunks):
    data.append({
        "id": i,
        "vector": chunk_vectors[i],
        "text": c["text"],
        "source": c["source"],
        "chunk_index": c["chunk_index"],
    })

client.insert(collection_name=COLLECTION, data=data)
print(f"✅ Stored {len(data)} chunks in Milvus collection '{COLLECTION}'.")

### 1.5 — Query! (the payoff)
Type a question. We embed it with the **same model**, ask Milvus for the closest chunks,
and print them with their **distance** (lower = closer/more similar, for L2).

👉 Change `QUESTION` and `TOP_K` and re-run to explore.

In [ ]:
QUESTION = "What is this document about?"   # 👈 ask something your docs can answer
TOP_K = 3                                    # 👈 how many chunks to retrieve

query_vector = embedder.encode([QUESTION])

results = client.search(
    collection_name=COLLECTION,
    data=query_vector,
    limit=TOP_K,
    output_fields=["text", "source", "chunk_index"],
)

print(f"Question: {QUESTION}\n")
for rank, hit in enumerate(results[0], start=1):
    print(f"--- Result #{rank}  (distance={hit['distance']:.4f}) ---")
    print(f"source: {hit['entity']['source']}  chunk#{hit['entity']['chunk_index']}")
    print(hit["entity"]["text"][:400])
    print()

---
## Phase 2 — Chunking strategy experiments

> ### 🎯 Goal of Phase 2
> Learn that **how you cut the text matters as much as any fancy model.** Retrieval
> quality depends heavily on chunking. You'll try three strategies and compare which
> returns the most relevant chunks for the same question:
>
> 1. **Fixed-size** (what we did in Phase 1) — simple but blindly cuts mid-sentence.
> 2. **Fixed-size with overlap** — repeats a little text between chunks so ideas that
>    straddle a boundary aren't lost.
> 3. **Structure-aware** — split on markdown headers / paragraphs so each chunk is a
>    coherent unit.
>
> The takeaway: there's no single "best" — you'll develop intuition for the trade-offs.


### 2.1 — Three chunkers
We define all three so you can switch between them with one variable.

In [ ]:
import re

def chunk_fixed(text, size=500, overlap=0):
    step = max(1, size - overlap)
    return [text[i:i + size] for i in range(0, len(text), step)]

def chunk_by_structure(text, max_size=800):
    # Split on markdown headers first, then fall back to paragraph splits if too big.
    sections = re.split(r"(?=^#{1,6}\s)", text, flags=re.MULTILINE)
    out = []
    for sec in sections:
        sec = sec.strip()
        if not sec:
            continue
        if len(sec) <= max_size:
            out.append(sec)
        else:
            for para in sec.split("\n\n"):
                if para.strip():
                    out.append(para.strip())
    return out

print("Defined: chunk_fixed (with optional overlap) and chunk_by_structure.")

### 2.2 — A reusable "build & query" helper
This rebuilds the whole index from a list of chunk-dicts and runs a query — so every
experiment below is a one-liner. It drops and recreates the collection each time
(clean slate), which is exactly why non-persistent storage is convenient here.

In [ ]:
def build_and_query(chunk_dicts, question, top_k=3, metric="L2"):
    texts = [c["text"] for c in chunk_dicts]
    vecs = embedder.encode(texts, show_progress_bar=False)

    if client.has_collection(COLLECTION):
        client.drop_collection(COLLECTION)
    client.create_collection(collection_name=COLLECTION, dimension=DIMENSION, metric_type=metric)

    rows = [{"id": i, "vector": vecs[i], "text": c["text"],
             "source": c["source"], "chunk_index": c["chunk_index"]}
            for i, c in enumerate(chunk_dicts)]
    client.insert(collection_name=COLLECTION, data=rows)

    q = embedder.encode([question])
    res = client.search(collection_name=COLLECTION, data=q, limit=top_k,
                        output_fields=["text", "source", "chunk_index"])
    print(f"[{len(rows)} chunks | metric={metric}]  Q: {question}")
    for rank, hit in enumerate(res[0], start=1):
        print(f"  #{rank} dist={hit['distance']:.4f}  {hit['entity']['source']}#{hit['entity']['chunk_index']}")
        print("     ", hit["entity"]["text"][:150].replace("\n", " "), "...")
    print()
    return res

def make_chunks(chunker, **kw):
    out = []
    for doc in documents:
        for i, piece in enumerate(chunker(doc["text"], **kw)):
            out.append({"text": piece, "source": doc["source"], "chunk_index": i})
    return out

print("Helpers ready: build_and_query(...) and make_chunks(...)")

### 2.3 — Compare the three strategies on the SAME question
Read the results below each other. Which strategy surfaces the most on-topic chunks?
Which has awkward mid-sentence cuts?

👉 Change `Q` to a real question about your docs.

In [ ]:
Q = "What is this document about?"   # 👈 use a real question

print("=== Strategy 1: fixed-size, no overlap ===")
build_and_query(make_chunks(chunk_fixed, size=500, overlap=0), Q)

print("=== Strategy 2: fixed-size WITH overlap ===")
build_and_query(make_chunks(chunk_fixed, size=500, overlap=100), Q)

print("=== Strategy 3: structure-aware (headers/paragraphs) ===")
build_and_query(make_chunks(chunk_by_structure, max_size=800), Q)

### 2.4 — Vary chunk size for the winning strategy
Small chunks = precise but may lack context. Large chunks = more context but noisier
and less precise. Find the sweet spot for *your* docs and question.

In [ ]:
for size in [200, 500, 1000]:
    print(f"=== fixed-size = {size}, overlap = {size//5} ===")
    build_and_query(make_chunks(chunk_fixed, size=size, overlap=size//5), Q)

> **What to notice (jot your own answers):**
> - Did overlap change which chunks came back? Did it help?
> - Did structure-aware chunks look more "complete" (start/end at natural boundaries)?
> - Smaller vs larger chunks — which gave more relevant results for your question?
>
> There is no universal winner — this intuition is the real lesson.


---
## Phase 2B — Semantic chunking + visualization (adapted from your trainer's notebook)

> ### 🎯 Goal of Phase 2B
> The chunkers in Phase 2 cut text by **shape** (character count, paragraphs). This phase
> introduces **semantic chunking** — cutting by **meaning**. Instead of "every 500 chars,"
> it finds the points where the *topic actually shifts* and cuts there, so each chunk is a
> coherent idea.
>
> How it works (3 moves, all local — no API keys):
> 1. **Segment** the doc into sentences.
> 2. **Embed** each sentence and measure how different each sentence is from the previous one
>    (cosine distance). Big jumps = topic changes = **breakpoints**.
> 3. **Merge** sentences between breakpoints into chunks.
>
> Then you'll **visualize** all chunks as points in 2D (UMAP) and color them by cluster, so you
> can literally *see* which chunks are about similar things. Finally, we feed the semantic
> chunks into Milvus and retrieve — so this plugs into your existing pipeline.
>
> This is a simplified, keys-free version of the approach your trainer demonstrated
> (their version used LlamaIndex + a Groq LLM for cluster labels; we keep it to local models).


### 2B.1 — Install the extra libraries for visualization
`umap-learn` + `matplotlib` for the 2D plot, `nltk` for good sentence splitting, `scikit-learn`
for clustering. Run once. (Restart not needed.)

In [ ]:
!pip install -q -U umap-learn matplotlib scikit-learn nltk
import nltk
nltk.download("punkt", quiet=True)
nltk.download("punkt_tab", quiet=True)
print("✅ Visualization libraries ready.")

### 2B.2 — Step 1: split each doc into sentences
Semantic chunking needs *fine-grained* candidate split points, so we start at the sentence
level. We use NLTK's Punkt tokenizer (handles abbreviations better than a naive `.split('.')`).
`min_length` drops tiny fragments.

In [ ]:
from nltk.tokenize import sent_tokenize

def split_into_sentences(text, min_length=30):
    sentences = sent_tokenize(text)
    return [s.strip() for s in sentences if len(s.strip()) >= min_length]

# segment each uploaded document
doc_sentences = {d["source"]: split_into_sentences(d["text"]) for d in documents}
for src, sents in doc_sentences.items():
    print(f"{src}: {len(sents)} sentences")
print("\nExample sentence:", doc_sentences[documents[0]["source"]][0][:200])

### 2B.3 — Step 2: embed sentences & find topic-shift breakpoints
We embed every sentence, then compute the cosine distance between each sentence and the one
before it. A **large** distance means the topic jumped — that's a breakpoint. We keep the
top `(1 - threshold)` fraction of jumps as breakpoints.

👉 `THRESHOLD` is your experiment knob: higher (e.g. 0.95) = fewer, bigger chunks; lower
(e.g. 0.80) = more, smaller chunks.

In [ ]:
import numpy as np
from scipy.spatial.distance import cosine

# reuse a real embedding model (make sure `embedder` is a valid sentence model from Phase 1.3)
def find_breakpoints(sentences, threshold=0.90):
    if len(sentences) < 2:
        return np.array([]), np.array([])
    embs = embedder.encode(sentences, show_progress_bar=False)
    # distance between consecutive sentences
    dists = np.array([cosine(embs[i], embs[i-1]) for i in range(1, len(embs))])
    cutoff = np.percentile(dists, 100 * threshold)
    breakpoints = np.argwhere(dists >= cutoff).ravel() + 1  # index of sentence that STARTS a new chunk
    return breakpoints, dists

THRESHOLD = 0.90   # 👈 experiment: 0.80 (more chunks) ... 0.95 (fewer chunks)

target_doc = documents[0]["source"]      # 👈 which doc to analyze
sents = doc_sentences[target_doc]
breaks, dists = find_breakpoints(sents, THRESHOLD)
print(f"{target_doc}: {len(sents)} sentences → {len(breaks)} breakpoints (threshold={THRESHOLD})")
print("Breakpoint sentence indices:", breaks.tolist())

### 2B.4 — Step 3: merge sentences into semantic chunks
Group the sentences between consecutive breakpoints. Each resulting chunk is a run of
sentences that are about the same thing.

In [ ]:
def build_semantic_chunks(sentences, breakpoints):
    bounds = [0] + sorted(breakpoints.tolist()) + [len(sentences)]
    chunks_out = []
    for start, end in zip(bounds[:-1], bounds[1:]):
        piece = " ".join(sentences[start:end]).strip()
        if piece:
            chunks_out.append(piece)
    return chunks_out

# build semantic chunks for ALL docs (so we can store + visualize everything)
semantic_chunks = []
for doc in documents:
    s = doc_sentences[doc["source"]]
    bps, _ = find_breakpoints(s, THRESHOLD)
    for i, piece in enumerate(build_semantic_chunks(s, bps)):
        semantic_chunks.append({"text": piece, "source": doc["source"], "chunk_index": i})

print(f"Total semantic chunks across all docs: {len(semantic_chunks)}")
print("\n--- Example semantic chunk ---")
print(semantic_chunks[0]["text"][:400], "...")

### 2B.5 — Retrieve semantic chunks from Milvus (plug into your pipeline)
Reuse the `build_and_query` helper from Phase 2.2 to store these semantic chunks in Milvus
and run a query. Compare the results to the fixed-size / structure-aware runs from Phase 2.

In [ ]:
Q = "What is this document about?"   # 👈 your question

print("=== SEMANTIC chunking → Milvus retrieval ===")
build_and_query(semantic_chunks, Q, top_k=3)

### 2B.6 — Visualize the chunks in 2D (UMAP)
Every semantic chunk is a high-dimensional vector. UMAP squashes those to 2D so we can plot
them. We then cluster nearby chunks and color them. **Chunks about the same topic land near
each other** — that's the intuition you're building.

(This is the keys-free version of your trainer's UMAP/dendrogram visualization — we skip the
LLM cluster-labeling that needed a Groq key.)

In [ ]:
import umap
import matplotlib.pyplot as plt
from sklearn.cluster import AgglomerativeClustering

# 1) embed all semantic chunks
chunk_texts = [c["text"] for c in semantic_chunks]
chunk_embs = np.array(embedder.encode(chunk_texts, show_progress_bar=False))

# 2) reduce to 2D
n = len(chunk_embs)
reducer = umap.UMAP(n_components=2, n_neighbors=min(5, max(2, n - 1)),
                    min_dist=0.0, metric="cosine", random_state=42)
reduced = reducer.fit_transform(chunk_embs)

# 3) cluster (choose a sensible cluster count)
n_clusters = min(6, n)
labels = AgglomerativeClustering(n_clusters=n_clusters).fit_predict(chunk_embs)

# 4) plot
plt.figure(figsize=(10, 7))
scatter = plt.scatter(reduced[:, 0], reduced[:, 1], c=labels, cmap="tab10", s=80, alpha=0.85)
plt.colorbar(scatter, label="cluster")
# annotate each point with source + chunk index
for i, c in enumerate(semantic_chunks):
    plt.annotate(f'{c["source"][:6]}#{c["chunk_index"]}', (reduced[i, 0], reduced[i, 1]),
                 fontsize=7, alpha=0.7)
plt.title(f"Semantic chunks in 2D (UMAP) — {n} chunks, {n_clusters} clusters")
plt.xlabel("UMAP-1"); plt.ylabel("UMAP-2")
plt.tight_layout()
plt.show()

print("Points close together = semantically similar chunks.")
print("This is WHY retrieval works: your question lands near the chunks that answer it.")

> **What to notice in Phase 2B:**
> - Do semantic chunks retrieved from Milvus look more *self-contained* than fixed-size ones?
> - In the 2D plot, do chunks from the same section/topic cluster together?
> - Raise/lower `THRESHOLD` and re-run 2B.3→2B.6: fewer big chunks vs. many small ones — which
>   retrieves better for your question?
>
> **Concept:** semantic chunking trades simplicity for coherence. It costs an extra embedding
> pass (you embed every sentence), but chunks respect meaning instead of arbitrary boundaries.


---
## Phase 3 — Vector DB schema design (Milvus)

> ### 🎯 Goal of Phase 3
> This is the phase your trainer specifically asked about. So far we let Milvus
> auto-build a minimal schema. Now you'll **design the schema yourself** and feel why
> it matters. A vector DB collection is like a database table — you decide its columns,
> types, primary key, the index on the vector, and the distance metric.
>
> You'll run three experiments and record pros/cons:
> - **Experiment A — Metadata richness:** minimal fields vs. rich fields, and how rich
>   metadata unlocks **filtered search** (e.g. "only search within doc1.md").
> - **Experiment B — Distance metric:** `L2` vs. `COSINE` — how "similarity" is defined
>   changes your ranking.
> - **Experiment C — Index type:** `FLAT` (exact) vs. `HNSW` (fast approximate) — the
>   core scale trade-off (mostly academic on 3 docs, but you'll see *where* it matters).
>
> End result: a pros/cons decision table and a schema you choose to carry forward.


### 3.1 — Build a collection with an EXPLICIT schema
Instead of quick-setup, we declare every field. Notice we add real metadata columns:
`source`, `section`, `chunk_index`, `char_count`. These are what make filtered search possible.

In [ ]:
from pymilvus import DataType

def build_explicit(chunk_dicts, metric="COSINE", index_type="FLAT", collection="rag_rich"):
    vecs = embedder.encode([c["text"] for c in chunk_dicts], show_progress_bar=False)

    if client.has_collection(collection):
        client.drop_collection(collection)

    schema = client.create_schema(auto_id=False, enable_dynamic_field=False)
    schema.add_field("id", DataType.INT64, is_primary=True)
    schema.add_field("vector", DataType.FLOAT_VECTOR, dim=DIMENSION)
    schema.add_field("text", DataType.VARCHAR, max_length=4000)
    schema.add_field("source", DataType.VARCHAR, max_length=256)
    schema.add_field("section", DataType.VARCHAR, max_length=512)
    schema.add_field("chunk_index", DataType.INT64)
    schema.add_field("char_count", DataType.INT64)

    index_params = client.prepare_index_params()
    index_params.add_index(field_name="vector", index_type=index_type, metric_type=metric)

    client.create_collection(collection_name=collection, schema=schema, index_params=index_params)

    rows = []
    for i, c in enumerate(chunk_dicts):
        first_line = c["text"].strip().split("\n", 1)[0][:200]
        rows.append({
            "id": i, "vector": vecs[i], "text": c["text"],
            "source": c["source"], "section": first_line,
            "chunk_index": c["chunk_index"], "char_count": len(c["text"]),
        })
    client.insert(collection_name=collection, data=rows)
    print(f"✅ Built '{collection}' with explicit schema | metric={metric} index={index_type} | {len(rows)} rows")
    return collection

rich_col = build_explicit(make_chunks(chunk_by_structure, max_size=800))

### 3.2 — Experiment A: filtered search (only possible with rich metadata)
Because we stored `source` as a real field, we can restrict the search to one file.
This is a superpower minimal schemas don't have. Try changing the filter.

In [ ]:
Q = "What is this document about?"   # 👈 your question
q = embedder.encode([Q])

# pick one of your uploaded files to filter on:
target_source = documents[0]["source"]
print("Filtering to source ==", target_source, "\n")

res = client.search(
    collection_name=rich_col, data=q, limit=3,
    filter=f'source == "{target_source}"',          # 👈 metadata filter
    output_fields=["text", "source", "section", "char_count"],
)
for rank, hit in enumerate(res[0], start=1):
    e = hit["entity"]
    print(f"#{rank} dist={hit['distance']:.4f} | {e['source']} | section='{e['section'][:60]}' | {e['char_count']} chars")
    print("   ", e["text"][:150].replace("\n", " "), "...\n")

### 3.3 — Experiment B: distance metric (L2 vs COSINE)
Same data, same question — only the metric changes. Compare the ranking and the raw
distance numbers. `COSINE` measures angle (direction) between vectors; `L2` measures
straight-line distance. For text embeddings, `COSINE` is a very common default.

In [ ]:
Q = "What is this document about?"
q = embedder.encode([Q])

for metric in ["L2", "COSINE"]:
    col = build_explicit(make_chunks(chunk_by_structure, max_size=800),
                         metric=metric, collection=f"rag_{metric.lower()}")
    res = client.search(collection_name=col, data=q, limit=3,
                        output_fields=["text", "source", "chunk_index"])
    print(f"--- metric={metric} ---")
    for rank, hit in enumerate(res[0], start=1):
        print(f"  #{rank} score={hit['distance']:.4f}  {hit['entity']['source']}#{hit['entity']['chunk_index']}")
    print()

### 3.4 — Experiment C: index type (FLAT vs HNSW)
`FLAT` = brute-force exact search (checks every vector — perfect recall, slow at scale).
`HNSW` = graph-based approximate search (much faster at millions of vectors, tiny recall
risk). On a few hundred chunks results are usually identical — the point is to understand
*when* you'd switch (large collections).

In [ ]:
Q = "What is this document about?"
q = embedder.encode([Q])

for idx in ["FLAT", "HNSW"]:
    col = build_explicit(make_chunks(chunk_by_structure, max_size=800),
                         metric="COSINE", index_type=idx, collection=f"rag_idx_{idx.lower()}")
    res = client.search(collection_name=col, data=q, limit=3,
                        output_fields=["source", "chunk_index"])
    print(f"--- index={idx} ---")
    for rank, hit in enumerate(res[0], start=1):
        print(f"  #{rank} score={hit['distance']:.4f}  {hit['entity']['source']}#{hit['entity']['chunk_index']}")
    print()

### 3.5 — Pros / cons decision table
A starting matrix. Fill the last column with what YOU observed on your own docs.

| Decision | Option | Pros | Cons | What I observed |
|---|---|---|---|---|
| **Metadata** | Minimal (`id,vector,text`) | Simple, small, fast to set up | No filtering, no context fields | |
| | Rich (`+source,section,...`) | Filtered search, richer results, debuggable | More storage, must plan fields upfront | |
| **Metric** | `L2` | Intuitive straight-line distance | Sensitive to vector magnitude | |
| | `COSINE` | Standard for text; direction-based | Slightly less intuitive numbers | |
| **Index** | `FLAT` | Exact, perfect recall, zero tuning | Slow on huge datasets | |
| | `HNSW` | Very fast at scale | Approximate; memory + tuning knobs | |
| **Primary key** | `auto_id=True` | No key management | IDs not meaningful | |
| | Your own IDs | Meaningful, updatable | You must ensure uniqueness | |

> **Recommendation to carry forward for learning:** rich metadata + `COSINE` + `FLAT`.
> It gives you filtering and correct-by-construction results; `FLAT` is fine until you
> have hundreds of thousands of vectors.


---
## Phase 4 — Embedding & retrieval experiments

> ### 🎯 Goal of Phase 4
> Now that the schema is solid, tune the two remaining quality levers:
>
> 1. **Embedding model** — a better/bigger model can dramatically improve which chunks
>    match. You'll swap models and compare. **Key gotcha:** different models output
>    different vector dimensions, so the collection must be recreated to match — you'll
>    see this handled automatically.
> 2. **Top-K** — how many chunks you retrieve. Too few risks missing the answer; too
>    many adds noise (and, later, cost when an LLM reads them).
>
> You'll also combine everything: a rich-schema, filtered, model-of-your-choice search.


### 4.1 — Swap the embedding model and compare
We reload the embedder, recompute `DIMENSION`, and rebuild. Watch how the top results
(and the distances) shift between models.

👉 Try each model in `MODELS_TO_TRY`. The first is smallest/fastest; later ones are stronger.

In [ ]:
MODELS_TO_TRY = [
    "all-MiniLM-L6-v2",          # 384-dim, tiny & fast
    "BAAI/bge-small-en-v1.5",    # 384-dim, strong small model
    # "all-mpnet-base-v2",       # 768-dim, higher quality, slower (uncomment to try)
]

Q = "What is this document about?"   # 👈 your question
base_chunks = make_chunks(chunk_by_structure, max_size=800)

for model_name in MODELS_TO_TRY:
    m = SentenceTransformer(model_name, device=DEVICE)
    dim = m.get_sentence_embedding_dimension()
    vecs = m.encode([c["text"] for c in base_chunks], show_progress_bar=False)

    col = "rag_model_test"
    if client.has_collection(col):
        client.drop_collection(col)
    client.create_collection(collection_name=col, dimension=dim, metric_type="COSINE")
    client.insert(collection_name=col, data=[
        {"id": i, "vector": vecs[i], "text": c["text"], "source": c["source"], "chunk_index": c["chunk_index"]}
        for i, c in enumerate(base_chunks)
    ])

    qv = m.encode([Q])
    res = client.search(collection_name=col, data=qv, limit=3,
                        output_fields=["source", "chunk_index", "text"])
    print(f"=== {model_name} (dim={dim}) ===")
    for rank, hit in enumerate(res[0], start=1):
        print(f"  #{rank} score={hit['distance']:.4f}  {hit['entity']['source']}#{hit['entity']['chunk_index']}")
        print("     ", hit["entity"]["text"][:120].replace("\n"," "), "...")
    print()

### 4.2 — Vary Top-K
See how many results you'd need to actually capture the answer. Look at where the
distance scores "fall off a cliff" — beyond that, results are usually noise.

In [ ]:
Q = "What is this document about?"
# rebuild with our recommended setup (MiniLM + rich schema + cosine + flat)
embedder = SentenceTransformer("all-MiniLM-L6-v2", device=DEVICE)
DIMENSION = embedder.get_sentence_embedding_dimension()
col = build_explicit(make_chunks(chunk_by_structure, max_size=800), metric="COSINE", collection="rag_topk")

qv = embedder.encode([Q])
res = client.search(collection_name=col, data=qv, limit=8, output_fields=["source", "chunk_index"])
print(f"Top-8 for: {Q}\n(watch where scores drop off)\n")
for rank, hit in enumerate(res[0], start=1):
    print(f"  #{rank} score={hit['distance']:.4f}  {hit['entity']['source']}#{hit['entity']['chunk_index']}")

> **What to notice:**
> - Did a stronger embedding model return more on-topic chunks or change the order?
> - At what K do the results stop being relevant? That's a good default Top-K for your data.


---
## Phase 5 — Redo it with LangChain, then decide

> ### 🎯 Goal of Phase 5
> You've now built RAG "by hand" and understand every step. LangChain is a framework
> that does the same steps for you with less code. The goal here is **an informed
> comparison** so *you* can decide: LangChain or raw `pymilvus`?
>
> You'll rebuild the pipeline (load → split → embed → store in Milvus → retrieve) using
> LangChain's building blocks, then read a side-by-side pros/cons summary.
>
> Ask yourself while running it: *Is the shorter code worth not seeing what's happening
> under the hood?*


### 5.1 — The same pipeline in LangChain
Notice how chunking, embedding, storage, and retrieval collapse into a few objects.
LangChain uses the same Milvus Lite `.db` file underneath.

In [ ]:
from langchain_community.document_loaders import TextLoader
from langchain.text_splitter import RecursiveCharacterTextSplitter
from langchain_huggingface import HuggingFaceEmbeddings
from langchain_milvus import Milvus
import glob

# 1) Load
lc_docs = []
for path in glob.glob("docs/*.md"):
    lc_docs.extend(TextLoader(path, encoding="utf-8").load())

# 2) Split (LangChain's recursive splitter ~ our structure-aware chunker)
splitter = RecursiveCharacterTextSplitter(chunk_size=800, chunk_overlap=100)
splits = splitter.split_documents(lc_docs)
print(f"LangChain produced {len(splits)} chunks.")

# 3) Embed (same local model, wrapped by LangChain)
lc_embeddings = HuggingFaceEmbeddings(model_name="all-MiniLM-L6-v2")

# 4) Store in Milvus Lite (one call does embed + create + insert)
vectorstore = Milvus.from_documents(
    documents=splits,
    embedding=lc_embeddings,
    connection_args={"uri": "milvus_langchain.db"},
    collection_name="rag_langchain",
    drop_old=True,
)
print("✅ Built LangChain + Milvus vector store.")

### 5.2 — Retrieve with LangChain
`as_retriever` gives you a reusable retriever object. `k` is Top-K.

In [ ]:
Q = "What is this document about?"   # 👈 your question

retriever = vectorstore.as_retriever(search_kwargs={"k": 3})
hits = retriever.invoke(Q)

print(f"Question: {Q}\n")
for rank, doc in enumerate(hits, start=1):
    print(f"--- Result #{rank} ---")
    print("source:", doc.metadata.get("source"))
    print(doc.page_content[:300], "...\n")

### 5.3 — LangChain vs. raw pymilvus — decide for yourself

| | Raw `pymilvus` (Phases 1–4) | LangChain (Phase 5) |
|---|---|---|
| **Lines of code** | More — you write each step | Fewer — steps are bundled |
| **Visibility** | You see load/chunk/embed/store/query explicitly | Steps hidden inside objects |
| **Control over schema** | Full (Phase 3 explicit schema, filters, index, metric) | Limited / abstracted by default |
| **Swapping components** | Manual but transparent | Easy (swap loader/splitter/embeddings/store) |
| **Learning value** | High — nothing hidden | Lower — convenient but opaque |
| **Production speed** | Slower to write, precise | Faster to build, ecosystem of loaders/retrievers |
| **Best when** | You need control, or you're learning | You want to move fast / prototype |

> ### Your decision (write it down)
> - For **learning the concepts**: raw `pymilvus` wins — you saw every moving part.
> - For **building an app quickly**: LangChain wins — less boilerplate, batteries included.
> - Many teams learn raw first (like you did), then use LangChain (or LlamaIndex) in
>   production while understanding what it does underneath.
>
> **You've now done both — you're equipped to choose.** 🎉


---
## Where to go next (optional, not built here)
- **Add generation (full RAG):** feed the retrieved chunks to an LLM to actually *answer*
  the question (needs an API key). This was intentionally left out per your plan.
- **Persistence:** mount Google Drive and point the `.db` file there to survive restarts.
- **Evaluation:** measure retrieval quality with a set of question→expected-chunk pairs.
- **Hybrid search / re-ranking:** combine keyword + vector search, then re-rank results.
